# 📊 Análise do Mercado de Trabalho em Tecnologia da Informação no Brasil

**Aluno:** Piersilvio de Carvalho Orlandini  
**Disciplina:** Linguagem de Programação — Análise e Visualização de Dados com Python  
**Tema:** 25 — Mercado de TI no Brasil (2015–2024)  
**Avaliação:** G2


---
## 1. Introdução e Contextualização

A Tecnologia da Informação (TI) é um dos setores que mais cresceram na economia brasileira na última década. Impulsionado pela transformação digital, crescimento do e-commerce, fintechs e pela aceleração provocada pela pandemia de COVID-19, o mercado de trabalho em TI passou por profundas transformações entre 2015 e 2024.

Este projeto tem como objetivo **analisar tendências do mercado de trabalho em TI no Brasil**, investigando:

- Quais cargos possuem maior demanda?
- Quais tecnologias aparecem com maior frequência?
- Houve crescimento salarial ao longo do tempo?
- Existem diferenças regionais relevantes?
- Quais áreas de TI possuem maior crescimento?
- Qual nível de experiência é mais exigido?


---
## 2. Importação das Bibliotecas


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine, text
import warnings

warnings.filterwarnings('ignore')

# Configurações visuais
sns.set_theme(style='darkgrid', palette='Blues_d')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

print('✅ Bibliotecas importadas com sucesso!')
print(f'   pandas   {pd.__version__}')
print(f'   numpy    {np.__version__}')


---
## 3. Leitura e Explicação da Base de Dados


In [ ]:
# Leitura do dataset
df = pd.read_csv('../dados/simulacao_mercado_ti_brasil.csv', parse_dates=['data'])

print(f'✅ Dataset carregado com sucesso!')
print(f'   Linhas: {df.shape[0]:,}')
print(f'   Colunas: {df.shape[1]}')
print(f'   Período: {df["ano"].min()} a {df["ano"].max()}')


In [ ]:
# Visualizar as primeiras linhas
df.head(10)


In [ ]:
# Tipos de dados e valores únicos por coluna
print('=== Tipos de Dados ===')
print(df.dtypes)
print()
print('=== Valores Únicos por Coluna ===')
for col in df.columns:
    n = df[col].nunique()
    if n <= 20:
        print(f'  {col} ({n}): {df[col].unique().tolist()}')
    else:
        print(f'  {col}: {n} valores únicos')


**Descrição das colunas:**

| Coluna | Tipo | Descrição |
|---|---|---|
| `ano` | int | Ano da vaga (2015–2024) |
| `mes` | int | Mês da vaga (1–12) |
| `data` | datetime | Data de referência |
| `regiao` | str | Região do Brasil |
| `uf` | str | Unidade Federativa |
| `cidade` | str | Cidade |
| `cargo` | str | Nome do cargo |
| `senioridade` | str | Júnior, Pleno ou Sênior |
| `tecnologia` | str | Tecnologia exigida |
| `modalidade` | str | Presencial, Híbrido ou Remoto |
| `salario_medio` | float | Faixa salarial média (R$) |
| `quantidade_vagas` | int | Número de vagas |
| `empresa_setor` | str | Segmento da empresa |
| `nivel_demanda` | str | Baixo, Médio, Alto, Crítico |


---
## 4. Limpeza e Preparação dos Dados


In [ ]:
# Verificar valores nulos
print('=== Valores Nulos por Coluna ===')
nulos = df.isnull().sum()
print(nulos)
print(f'\nTotal de valores nulos: {nulos.sum()}')


In [ ]:
# Verificar duplicatas
dups = df.duplicated().sum()
print(f'Linhas duplicadas: {dups}')

# Verificar valores negativos em colunas numéricas
print(f'Salários negativos: {(df["salario_medio"] < 0).sum()}')
print(f'Vagas negativas: {(df["quantidade_vagas"] < 0).sum()}')


In [ ]:
# Verificar consistência das colunas categóricas
cols_cat = ['regiao', 'senioridade', 'modalidade', 'nivel_demanda']
for col in cols_cat:
    print(f'{col}: {sorted(df[col].unique())}')


In [ ]:
# Estatísticas descritivas das variáveis numéricas
df[['salario_medio', 'quantidade_vagas']].describe().round(2)


---
## 5. Engenharia de Atributos


In [ ]:
# Criar novas colunas úteis para análise
df['data'] = pd.to_datetime(df['data'])
df['ano_mes'] = df['data'].dt.to_period('M').astype(str)
df['trimestre'] = df['data'].dt.quarter
df['semestre'] = df['data'].dt.month.apply(lambda x: 1 if x <= 6 else 2)

# Faixa salarial
bins = [0, 5000, 10000, 15000, 25000, 100000]
labels = ['Até R$5k', 'R$5k–10k', 'R$10k–15k', 'R$15k–25k', 'Acima R$25k']
df['faixa_salarial'] = pd.cut(df['salario_medio'], bins=bins, labels=labels)

# Ordem de senioridade
ordem_sen = {'Júnior': 1, 'Pleno': 2, 'Sênior': 3}
df['ordem_senioridade'] = df['senioridade'].map(ordem_sen)

print('✅ Engenharia de atributos concluída!')
print(f'   Novas colunas: ano_mes, trimestre, semestre, faixa_salarial, ordem_senioridade')
df[['data', 'ano_mes', 'trimestre', 'semestre', 'faixa_salarial', 'senioridade', 'ordem_senioridade']].head()


---
## 6. KPIs — Indicadores-Chave de Desempenho


In [ ]:
total_vagas     = int(df['quantidade_vagas'].sum())
cargo_top       = df.groupby('cargo')['quantidade_vagas'].sum().idxmax()
tec_top         = df.groupby('tecnologia')['quantidade_vagas'].sum().idxmax()
salario_medio   = df['salario_medio'].mean()
regiao_top      = df.groupby('regiao')['quantidade_vagas'].sum().idxmax()
modalidade_top  = df.groupby('modalidade')['quantidade_vagas'].sum().idxmax()

print('=' * 55)
print('         📊 KPIs — MERCADO DE TI NO BRASIL')
print('=' * 55)
print(f'  Total de Vagas              : {total_vagas:,}'.replace(',', '.'))
print(f'  Cargo Mais Demandado        : {cargo_top}')
print(f'  Tecnologia Mais Requisitada : {tec_top}')
print(f'  Salário Médio Nacional      : R$ {salario_medio:,.2f}'.replace(',', '.'))
print(f'  Região com Mais Vagas       : {regiao_top}')
print(f'  Modalidade Predominante     : {modalidade_top}')
print('=' * 55)


---
## 7. Análise Exploratória


### 7.1 Evolução Temporal das Vagas


In [ ]:
ev_anual = df.groupby('ano')['quantidade_vagas'].sum().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(ev_anual['ano'], ev_anual['quantidade_vagas'],
              color=sns.color_palette('Blues_d', len(ev_anual)))

for bar, val in zip(bars, ev_anual['quantidade_vagas']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}'.replace(',', '.'), ha='center', fontsize=9)

ax.set_title('Evolução Anual do Total de Vagas em TI (2015–2024)', fontsize=14, fontweight='bold')
ax.set_xlabel('Ano')
ax.set_ylabel('Total de Vagas')
plt.tight_layout()
plt.savefig('../imagens/evolucao_anual.png', dpi=150, bbox_inches='tight')
plt.show()


### 7.2 Distribuição Regional


In [ ]:
reg_df = df.groupby('regiao')['quantidade_vagas'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras
sns.barplot(x=reg_df.index, y=reg_df.values, palette='Blues_d', ax=axes[0])
axes[0].set_title('Total de Vagas por Região', fontweight='bold')
axes[0].set_xlabel('Região')
axes[0].set_ylabel('Total de Vagas')

# Pizza
axes[1].pie(reg_df.values, labels=reg_df.index, autopct='%1.1f%%',
            colors=sns.color_palette('Blues_d', len(reg_df)))
axes[1].set_title('Distribuição Percentual por Região', fontweight='bold')

plt.tight_layout()
plt.savefig('../imagens/distribuicao_regional.png', dpi=150, bbox_inches='tight')
plt.show()


### 7.3 Ranking de Cargos e Tecnologias


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cargo_df = df.groupby('cargo')['quantidade_vagas'].sum().sort_values(ascending=True)
cargo_df.plot(kind='barh', ax=axes[0], color=sns.color_palette('Blues_d', len(cargo_df)))
axes[0].set_title('Total de Vagas por Cargo', fontweight='bold')
axes[0].set_xlabel('Total de Vagas')

tec_df = df.groupby('tecnologia')['quantidade_vagas'].sum().sort_values(ascending=True)
tec_df.plot(kind='barh', ax=axes[1], color=sns.color_palette('Greens_d', len(tec_df)))
axes[1].set_title('Total de Vagas por Tecnologia', fontweight='bold')
axes[1].set_xlabel('Total de Vagas')

plt.tight_layout()
plt.savefig('../imagens/cargos_tecnologias.png', dpi=150, bbox_inches='tight')
plt.show()


### 7.4 Evolução Salarial


In [ ]:
sal_ano = df.groupby('ano')['salario_medio'].mean().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(sal_ano['ano'], sal_ano['salario_medio'], marker='o', linewidth=2.5,
        color='#0ea5e9', markersize=8)
ax.fill_between(sal_ano['ano'], sal_ano['salario_medio'], alpha=0.15, color='#0ea5e9')

for _, row in sal_ano.iterrows():
    ax.annotate(f"R$ {row['salario_medio']:,.0f}".replace(',', '.'),
                (row['ano'], row['salario_medio']),
                textcoords='offset points', xytext=(0, 10), ha='center', fontsize=8)

ax.set_title('Evolução do Salário Médio em TI por Ano (2015–2024)', fontsize=14, fontweight='bold')
ax.set_xlabel('Ano')
ax.set_ylabel('Salário Médio (R$)')
plt.tight_layout()
plt.savefig('../imagens/evolucao_salarial.png', dpi=150, bbox_inches='tight')
plt.show()


### 7.5 Heatmap — Região vs Cargo


In [ ]:
pivot = df.pivot_table(index='regiao', columns='cargo',
                        values='quantidade_vagas', aggfunc='sum', fill_value=0)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(pivot, annot=True, fmt='d', cmap='Blues', linewidths=0.5, ax=ax)
ax.set_title('Heatmap — Vagas por Região e Cargo', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../imagens/heatmap_regiao_cargo.png', dpi=150, bbox_inches='tight')
plt.show()


### 7.6 Dispersão: Salário vs Vagas (Correlação Estatística)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for cargo in df['cargo'].unique():
    sub = df[df['cargo'] == cargo]
    ax.scatter(sub['salario_medio'], sub['quantidade_vagas'],
               alpha=0.4, label=cargo, s=30)

ax.set_xlabel('Salário Médio (R$)')
ax.set_ylabel('Quantidade de Vagas')
ax.set_title('Dispersão: Salário Médio vs Quantidade de Vagas', fontsize=13, fontweight='bold')
ax.legend()

corr = df['salario_medio'].corr(df['quantidade_vagas'])
ax.text(0.02, 0.95, f'r de Pearson = {corr:.4f}', transform=ax.transAxes,
        fontsize=10, color='gray', va='top')

plt.tight_layout()
plt.savefig('../imagens/dispersao_salario_vagas.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Coeficiente de Correlação de Pearson: {corr:.4f}')


### 7.7 Modalidade de Trabalho ao Longo do Tempo


In [ ]:
modal_tempo = df.groupby(['ano', 'modalidade'])['quantidade_vagas'].sum().reset_index()
modal_pivot = modal_tempo.pivot(index='ano', columns='modalidade', values='quantidade_vagas').fillna(0)

fig, ax = plt.subplots(figsize=(12, 5))
modal_pivot.plot(ax=ax, marker='o', linewidth=2)
ax.set_title('Modalidade de Trabalho por Ano', fontsize=13, fontweight='bold')
ax.set_xlabel('Ano')
ax.set_ylabel('Total de Vagas')
ax.legend(title='Modalidade')
plt.tight_layout()
plt.savefig('../imagens/modalidade_tempo.png', dpi=150, bbox_inches='tight')
plt.show()


### 7.8 Análise de Senioridade e Salário


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Salário por senioridade
sal_sen = df.groupby('senioridade')['salario_medio'].mean().reindex(['Júnior', 'Pleno', 'Sênior'])
sal_sen.plot(kind='bar', ax=axes[0], color=['#38bdf8', '#06b6d4', '#0369a1'], edgecolor='none')
axes[0].set_title('Salário Médio por Senioridade', fontweight='bold')
axes[0].set_ylabel('Salário Médio (R$)')
axes[0].set_xticklabels(['Júnior', 'Pleno', 'Sênior'], rotation=0)

# Box plot
df_box = df[['senioridade', 'salario_medio']].copy()
ordem = ['Júnior', 'Pleno', 'Sênior']
sns.boxplot(data=df_box, x='senioridade', y='salario_medio',
            order=ordem, palette='Blues', ax=axes[1])
axes[1].set_title('Distribuição Salarial por Senioridade', fontweight='bold')
axes[1].set_ylabel('Salário Médio (R$)')

plt.tight_layout()
plt.savefig('../imagens/salario_senioridade.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 8. Persistência em Banco de Dados (SQLAlchemy + SQLite)


In [ ]:
import os
os.makedirs('../database', exist_ok=True)

engine = create_engine('sqlite:///../database/mercado_ti.db')
df.to_sql('mercado_ti', engine, if_exists='replace', index=False)
print('✅ Dados persistidos no banco SQLite!')

# Consulta de exemplo
with engine.connect() as conn:
    resultado = pd.read_sql(
        text('SELECT cargo, ROUND(AVG(salario_medio), 2) as salario_medio, SUM(quantidade_vagas) as total_vagas FROM mercado_ti GROUP BY cargo ORDER BY salario_medio DESC'),
        conn
    )
print('\n📊 Ranking por Cargo (via SQL):')
resultado


---
## 9. Interpretação dos Resultados

### Tendências identificadas:

1. **Crescimento do setor:** O mercado de TI cresceu consistentemente de 2015 a 2024, com aceleração notável a partir de 2020, impulsionada pela digitalização pós-pandemia.

2. **Tecnologias dominantes:** Python e SQL lideram as exigências técnicas, evidenciando a demanda por profissionais de dados. JavaScript e Java mantêm relevância no desenvolvimento.

3. **Concentração regional:** O Sudeste — principalmente SP e RJ — concentra a maior parte das vagas. Entretanto, o trabalho remoto tem redistribuído oportunidades para outras regiões.

4. **Evolução salarial:** Observa-se tendência de alta nos salários, especialmente para cargos seniores como DevOps e Cientista de Dados, que apresentam remuneração significativamente superior.

5. **Senioridade:** Profissionais Sênior recebem em média 2 a 3 vezes mais que Júniores, indicando grande valorização da experiência.

6. **Modalidade de trabalho:** O trabalho remoto cresceu exponencialmente após 2020, saindo de modalidade marginal para uma das principais formas de contratação.


---
## 10. Conclusão

Este projeto analisou o mercado de trabalho em TI no Brasil entre 2015 e 2024, utilizando um dataset com 4.440 registros distribuídos em 5 regiões, 5 cargos e 6 tecnologias diferentes.

**Principais achados:**
- O mercado de TI brasileiro é dinâmico e em expansão, com demanda crescente por profissionais qualificados;
- Python e SQL são as competências mais valorizadas, alinhadas com a expansão da área de dados;
- O Sudeste concentra as melhores oportunidades, mas o trabalho remoto democratiza o acesso;
- A qualificação profissional tem impacto direto e significativo na remuneração;
- O setor de Fintech lidera as contratações, seguido por Saúde e Educação.

**Recomendações para profissionais de TI:**
- Investir em Python, SQL e Cloud (AWS) para maior empregabilidade;
- Buscar especialização para atingir nível Sênior e maximizar remuneração;
- Considerar oportunidades remotas em empresas do Sudeste mesmo residindo em outras regiões.

---
*Piersilvio de Carvalho Orlandini — Avaliação G2 — Análise e Visualização de Dados com Python*
